# 07 Segmentation To Grasp

深度感知常常不是直接输出动作，而是先选择哪些点属于目标物体。

![Segmentation to grasp visual map](assets/figures/07_segmentation_to_grasp.png)

## Learning Objectives

- Trace how a segmentation mask changes the point cloud passed to grasp scoring.
- Explain false positives and false negatives as downstream manipulation errors.
- Distinguish perception quality from end-to-end task success.

## Checkpoint

- Run the mask example and identify how many points are treated as target points.
- Explain why a background point can corrupt a grasp score.
- Name one evaluation metric that checks manipulation impact, not just segmentation accuracy.

## Practice Task

Flip one mask value at a time and record which mistake damages the selected grasp most.

## Concept Map

![Colab concept image](assets/colab/07_segmentation_to_grasp_concept.png)

**Concept image.** A segmentation mask decides which points reach the grasp scorer, so perception errors become grasp errors.

In [ ]:
from pathlib import Path
import sys

COLAB_REPO_URL = "https://github.com/Hollis36/robotic-manipulation-learning.git"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    in_colab = "google.colab" in sys.modules
    if in_colab:
        import subprocess

        PROJECT_ROOT = Path("/content/robotic-manipulation-learning")
        if not PROJECT_ROOT.exists():
            # Equivalent command: git clone --depth 1 <repo> <target>
            subprocess.run(["git", "clone", "--depth", "1", COLAB_REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


In [ ]:
import numpy as np
from rml.grasp_scoring import score_antipodal_grasps

points = np.array([[-0.5, 0.0], [-0.45, 0.02], [0.45, -0.01], [0.5, 0.0], [0.0, 0.6], [1.2, 1.2]])
mask = np.array([True, True, True, True, False, False])
target_points = points[mask]
candidates = [
    {"name": "target_object", "center": [0.0, 0.0], "angle": 0.0, "width": 1.0},
    {"name": "background", "center": [0.0, 0.6], "angle": 0.0, "width": 0.5},
]
best = score_antipodal_grasps(target_points, candidates)[0]
len(target_points), best.name, round(best.score, 2)

## Result Figure

Plot all observed points, the selected mask points, and the center of the best grasp candidate.

The figure below is generated from the values computed in this notebook. Treat it as evidence from the code, not as a decorative illustration.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
fig, ax = plt.subplots()
ax.scatter(points[:, 0], points[:, 1], s=85, color='0.7', label='all points')
ax.scatter(target_points[:, 0], target_points[:, 1], s=95, label='selected mask points')
ax.scatter([best.center[0]], [best.center[1]], marker='x', s=140, label=f'best: {best.name}')
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x position')
ax.set_ylabel('y position')
ax.legend(frameon=False)
plt.show()

## Parameter Experiment

The next cell is marked with `COLAB_PARAMETER_EXPERIMENT` so it is easy to find in Colab. Flip one mask value at a time and print which grasp becomes best after each perception mistake.

In [ ]:
# COLAB_PARAMETER_EXPERIMENT
for flip_index in range(len(mask)):
    trial_mask = mask.copy()
    trial_mask[flip_index] = ~trial_mask[flip_index]
    trial_points = points[trial_mask]
    trial_best = score_antipodal_grasps(trial_points, candidates)[0]
    print('flipped_index=', flip_index, 'selected_points=', len(trial_points), 'best=', trial_best.name, 'score=', round(trial_best.score, 2))

## Reflection Prompt

哪一种 mask 错误最危险：漏掉目标点，还是加入背景点？请用 best grasp 的变化解释。

Exercise: 把一个 background point 错误加入 mask，解释下游 grasp score 会怎样受影响。